[Open in Colab](https://colab.research.google.com/github/BoydVonstocker/Introduction-to-Quantum-Computing/blob/main/Notebks/03%20-%20Superposition_and_Entanglement.ipynb)


# Notebook 03 — Superposition and Entanglement

Superposition lets one qubit hold a combination of $|0\rangle$ and $|1\rangle$. Entanglement correlates two or more qubits so that the joint state cannot be written as a product $|\psi\rangle\otimes|\phi\rangle$.


In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, entropy
from qiskit.visualization import plot_histogram, plot_bloch_multivector, array_to_latex, plot_state_qsphere
from qiskit_aer import AerSimulator


## Superposition is not a classical mixture

$|+\rangle = (|0\rangle+|1\rangle)/\sqrt{2}$ is a *coherent* combination. The relative phase is physical. A classical coin that is 50/50 heads or tails has no phase and cannot interfere.

Interference is easy to see: two Hadamards in a row return $|0\rangle$ every time, because the amplitudes of $|1\rangle$ cancel.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.h(0)
qc.measure(0, 0)
display(qc.draw("mpl"))
plot_histogram(AerSimulator().run(qc, shots=1024).result().get_counts())


## Relative phase changes interference

Insert a $Z$ between the two Hadamards. $HZH|0\rangle = |1\rangle$. The same two coins "flipped twice" now land on the opposite side because the phase of $|1\rangle$ flipped.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.z(0)
qc.h(0)
qc.measure(0, 0)
display(qc.draw("mpl"))
plot_histogram(AerSimulator().run(qc, shots=1024).result().get_counts())


## Product states versus entangled states

A **product state** factors:

$$
|0\rangle\otimes|+\rangle = |0\rangle\otimes\frac{|0\rangle+|1\rangle}{\sqrt{2}} = \frac{|00\rangle+|01\rangle}{\sqrt{2}}.
$$

An **entangled state** does not factor. The four Bell states are the standard examples:

$$
|\Phi^+\rangle = \frac{|00\rangle+|11\rangle}{\sqrt{2}}, \quad
|\Phi^-\rangle = \frac{|00\rangle-|11\rangle}{\sqrt{2}},
$$

$$
|\Psi^+\rangle = \frac{|01\rangle+|10\rangle}{\sqrt{2}}, \quad
|\Psi^-\rangle = \frac{|01\rangle-|10\rangle}{\sqrt{2}}.
$$


## Building $|\Phi^+\rangle$

Hadamard on the control, CNOT onto the target.


In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
display(bell.draw("mpl"))

sv = Statevector.from_instruction(bell)
display(array_to_latex(sv, prefix=r"$|\Phi^+\rangle = $"))
plot_state_qsphere(sv)


## Measurement correlations

Measure both qubits of $|\Phi^+\rangle$. You only ever see `00` or `11`, each about half the time. Seeing one bit tells you the other — not because a signal travelled, but because the joint state was correlated from the start.


In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
display(qc.draw("mpl"))
counts = AerSimulator().run(qc, shots=2048).result().get_counts()
print(counts)
plot_histogram(counts)


## Local Bloch vectors of a Bell pair

If you look at only one qubit of $|\Phi^+\rangle$, it is maximally mixed: its Bloch vector has length 0. Entanglement is hidden from any single-qubit picture.


In [ ]:
sv = Statevector.from_instruction(bell)
rho0 = partial_trace(sv, [1])
print("reduced density matrix of qubit 0:")
display(array_to_latex(rho0))
print("von Neumann entropy of qubit 0:", float(entropy(rho0)))
plot_bloch_multivector(sv)


## The other Bell states

Change the input bits before the Hadamard + CNOT pair.


In [ ]:
def bell_state(x=0, y=0):
    qc = QuantumCircuit(2)
    if y:
        qc.x(1)
    if x:
        qc.x(0)
    qc.h(0)
    qc.cx(0, 1)
    return qc

for x, y, name in [(0, 0, "Phi+"), (1, 0, "Psi+"), (0, 1, "Phi-"), (1, 1, "Psi-")]:
    sv = Statevector.from_instruction(bell_state(x, y))
    print(name, {k: round(v, 3) for k, v in sv.probabilities_dict().items()})


## Exercises

1. Prepare $|\Psi^-\rangle$ and confirm you only measure `01` and `10`.
2. Start with two qubits in $|++\rangle$. Is that state entangled? Check whether it factors and whether the reduced entropy is zero.
3. After creating $|\Phi^+\rangle$, apply $X$ to one qubit. Which Bell state do you get?


In [ ]:
# Exercise workspace
